# 📓 Notebook 3 — Prototipo del Videojuego: Mapa e Interfaz

**Sección 2 — Modelación y Gamificación** · Día 3 (tarde)

### Objetivo de hoy
Diseñar y prototipar la **interfaz del videojuego** que representa la simulación que construyeron esta mañana: un mapa/tablero interactivo con controles básicos.

Partimos de los **bocetos en papel** que dibujaron al inicio de la sesión (pantallas, controles, escenarios) y los convertimos en un prototipo funcional.


## 0. Conectemos con lo anterior

Esta mañana (Notebook 2) construyeron una simulación que calcula números — variables que suben y bajan. Ahora vamos a **verla**, no solo calcularla.

**✏️ Antes de programar, respondan mirando su boceto en papel:**

1. En su boceto, ¿cuántas pantallas o escenas dibujaron? ¿Qué información mostraba cada una?
2. ¿Cómo decidieron representar visualmente "más vegetación" o "más contaminación" en su boceto: con colores, números, íconos, tamaños?
3. ¿Qué controles (botones, sliders, menús) dibujaron para que el jugador tome decisiones?

*No hay respuesta incorrecta — la idea es comparar su diseño original con el prototipo de código que van a construir hoy.*


## 1. Mini-lección: ¿con qué construimos el videojuego?

Hay varias formas de hacer la interfaz de un videojuego en Python. Para este taller usamos **dos caminos**, según dónde estén trabajando:

| Herramienta | Dónde funciona | Ventaja |
|---|---|---|
| **Grid interactivo con `matplotlib` + `ipywidgets`** | Google Colab y Jupyter (¡sin instalar nada!) | Funciona directo en el notebook, ideal para prototipar rápido |
| **Pygame** | Solo en Python instalado localmente (no en Colab) | Ventanas de juego "de verdad", más control sobre gráficos y teclado |

**Hoy construimos el prototipo principal con `matplotlib` + `ipywidgets`** para que todo el equipo pueda avanzar sin importar si usan Colab o su computadora. Al final de este notebook hay una sección opcional con el mismo prototipo en **Pygame** por si alguien quiere probarlo en su computadora local.

Ambos caminos comparten la misma idea de diseño de juego:
1. Un **tablero/mapa** (grid) que representa visualmente el sistema.
2. **Controles** que el jugador usa para tomar decisiones.
3. Una **simulación** por debajo (¡la que ya construyeron!) que reacciona a esas decisiones.

**🤔 Antes de continuar:** ¿qué ventaja y qué desventaja le ven a usar `matplotlib`/`ipywidgets` (funciona en Colab, pero se ve más simple) frente a Pygame (más "de videojuego", pero no corre en Colab) para el proyecto final de SU equipo?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output
import json, os

print("Librerías cargadas: numpy, matplotlib, ipywidgets ✅")


## 2. El tablero: representando el mapa de la ciudad como una cuadrícula

Vamos a representar la zona de estudio como una **cuadrícula (grid)** de N×N celdas. Cada celda tiene un tipo de uso de suelo:

| Código | Tipo de celda | Color |
|---|---|---|
| `0` | Agua 💧 | Azul |
| `1` | Vegetación 🌳 | Verde |
| `2` | Urbano / vivienda 🏠 | Gris |
| `3` | Industrial 🏭 | Naranja |

Esta cuadrícula es el "boceto en papel" que dibujaron, pero ahora en código.


In [ ]:
TIPOS_CELDA = {
    0: {"nombre": "agua", "emoji": "💧", "color": "#3b82c4"},
    1: {"nombre": "vegetacion", "emoji": "🌳", "color": "#4caf6b"},
    2: {"nombre": "urbano", "emoji": "🏠", "color": "#9e9e9e"},
    3: {"nombre": "industrial", "emoji": "🏭", "color": "#e08a2c"},
}

CMAP = ListedColormap([TIPOS_CELDA[i]["color"] for i in range(len(TIPOS_CELDA))])

def mapa_inicial_ejemplo(n=10, semilla=7):
    """Genera un mapa inicial de ejemplo (10x10). En el Notebook 5 lo reemplazaremos
    por datos satelitales reales de su zona (NDVI / uso de suelo)."""
    rng = np.random.default_rng(semilla)
    grid = rng.choice([0, 1, 1, 1, 2, 2], size=(n, n))  # más probabilidad de vegetación/urbano que agua
    grid[0, :3] = 0   # una franja de agua en una esquina, como ejemplo (ej. un río)
    return grid

grid = mapa_inicial_ejemplo()
print("Mapa de ejemplo generado (10x10):")
print(grid)


In [ ]:
def dibujar_mapa(grid, titulo="Mapa de la ciudad"):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(grid, cmap=CMAP, vmin=0, vmax=len(TIPOS_CELDA) - 1)
    ax.set_title(titulo)
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.grid(color="white", linewidth=1.5)
    ax.tick_params(length=0, labelbottom=False, labelleft=False)
    plt.show()

dibujar_mapa(grid, "Mapa inicial de ejemplo")


### 🔍 Observen y reflexionen

**✏️ Comparen este mapa de ejemplo con su boceto en papel de la Sección 0:**

- ¿Se parece a lo que imaginaron? ¿Qué le falta para representar mejor su ciudad? *(reemplazar)*
- Solo tenemos 4 tipos de celda. ¿Su boceto necesitaba más categorías? ¿Cuáles agregarían? *(reemplazar)*


## 3. Conectemos el mapa con la simulación de esta mañana

Volvamos a traer la clase `EcosistemaUrbano` del Notebook 2 (o cárguenla desde `simulacion_ecosistema.py` si ya lo generaron). Vamos a hacer que **el mapa alimente las variables de la simulación**: por ejemplo, `vegetacion` será el % de celdas de tipo vegetación, y `construccion` será el % de celdas urbanas + industriales.

**🤔 Antes de correr el código — predicción:** si su equipo convierte una celda de vegetación en una celda industrial, ¿qué esperan que le pase a los indicadores `vegetacion`, `contaminacion` y `agua` de su simulación? Anoten su hipótesis antes de comprobarla en la Sección 4.


In [ ]:
# Si generaron simulacion_ecosistema.py en el Notebook 2, lo importamos directamente.
# Si no, definimos aquí una copia mínima para que este notebook funcione de forma independiente.

if os.path.exists("simulacion_ecosistema.py"):
    import importlib.util
    spec = importlib.util.spec_from_file_location("simulacion_ecosistema", "simulacion_ecosistema.py")
    sim_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sim_mod)
    EcosistemaUrbano = sim_mod.EcosistemaUrbano
    print("✅ Clase EcosistemaUrbano importada desde simulacion_ecosistema.py")
else:
    class EcosistemaUrbano:
        # Copia mínima de la clase del Notebook 2 (usada si no se encontró el archivo .py)
        def __init__(self, modelo, ruido=0.03):
            self.variables_info = {v["nombre"]: v for v in modelo["variables"]}
            self.relaciones = modelo["relaciones"]
            self.ruido = ruido
            self.estado = {v["nombre"]: float(v["valor_inicial"]) for v in modelo["variables"]}
            self.historial = {nombre: [valor] for nombre, valor in self.estado.items()}

        def _normalizar(self, nombre, valor):
            minimo, maximo = self.variables_info[nombre]["rango"]
            return 0.0 if maximo == minimo else (valor - minimo) / (maximo - minimo)

        def _clip(self, nombre, valor):
            minimo, maximo = self.variables_info[nombre]["rango"]
            return max(minimo, min(maximo, valor))

        def calcular_efectos(self):
            efectos = {nombre: 0.0 for nombre in self.estado}
            for r in self.relaciones:
                origen_norm = self._normalizar(r["origen"], self.estado[r["origen"]])
                efectos[r["destino"]] += r["signo"] * r["fuerza"] * origen_norm
            return efectos

        def paso(self, acciones=None):
            acciones = acciones or {}
            efectos = self.calcular_efectos()
            nuevo_estado = {}
            for nombre, valor in self.estado.items():
                minimo, maximo = self.variables_info[nombre]["rango"]
                escala = maximo - minimo
                cambio = efectos[nombre] * escala * 0.15 + acciones.get(nombre, 0.0) + np.random.normal(0, self.ruido) * escala
                nuevo_estado[nombre] = self._clip(nombre, valor + cambio)
            self.estado = nuevo_estado
            for nombre, valor in self.estado.items():
                self.historial[nombre].append(valor)
            return self.estado
    print("⚠️ Usando copia mínima de EcosistemaUrbano (no se encontró simulacion_ecosistema.py)")

# Cargamos el modelo del equipo (o el de ejemplo si no existe)
if os.path.exists("modelo_mental.json"):
    with open("modelo_mental.json", "r", encoding="utf-8") as f:
        modelo = json.load(f)
else:
    modelo = {
        "equipo": "Equipo de ejemplo", "zona_de_estudio": "Zona de ejemplo",
        "variables": [
            {"nombre": "vegetacion", "descripcion": "Cobertura vegetal", "unidad": "%", "rango": [0, 100], "valor_inicial": 60},
            {"nombre": "construccion", "descripcion": "Área construida", "unidad": "%", "rango": [0, 100], "valor_inicial": 30},
            {"nombre": "poblacion", "descripcion": "Habitantes (miles)", "unidad": "miles", "rango": [0, 500], "valor_inicial": 100},
            {"nombre": "contaminacion", "descripcion": "Contaminación", "unidad": "índice", "rango": [0, 100], "valor_inicial": 20},
            {"nombre": "agua", "descripcion": "Calidad de agua", "unidad": "índice", "rango": [0, 100], "valor_inicial": 70},
        ],
        "relaciones": [
            {"origen": "construccion", "destino": "vegetacion", "signo": -1, "fuerza": 0.4},
            {"origen": "construccion", "destino": "poblacion", "signo": 1, "fuerza": 0.3},
            {"origen": "poblacion", "destino": "contaminacion", "signo": 1, "fuerza": 0.3},
            {"origen": "vegetacion", "destino": "contaminacion", "signo": -1, "fuerza": 0.35},
            {"origen": "contaminacion", "destino": "agua", "signo": -1, "fuerza": 0.3},
            {"origen": "agua", "destino": "vegetacion", "signo": 1, "fuerza": 0.25},
        ],
    }

print(f"Modelo cargado — Equipo: {modelo['equipo']}")


In [ ]:
def variables_desde_grid(grid):
    """Convierte el mapa (grid) en valores de variables de la simulación."""
    total = grid.size
    return {
        "vegetacion": 100 * np.sum(grid == 1) / total,
        "construccion": 100 * np.sum((grid == 2) | (grid == 3)) / total,
        "agua": 100 * np.sum(grid == 0) / total,
    }

print("Variables calculadas a partir del mapa inicial:")
print(variables_desde_grid(grid))


## 4. Controles básicos del juego

Diseñamos 3 controles simples, tal como lo bocetaron en papel:

1. **Fila y columna**: sliders para elegir qué celda del mapa modificar.
2. **Acción**: un menú desplegable para elegir qué construir ahí (reforestar, construir vivienda, poner fábrica).
3. **Botón "Aplicar acción"**: ejecuta el cambio en el mapa.
4. **Botón "Avanzar turno"**: corre un paso de la simulación con el mapa actualizado.


In [ ]:
ACCIONES_DISPONIBLES = {
    "🌳 Reforestar": 1,
    "🏠 Construir vivienda": 2,
    "🏭 Poner fábrica": 3,
    "💧 Restaurar cuerpo de agua": 0,
}

estado_juego = {
    "grid": mapa_inicial_ejemplo(),
    "eco": EcosistemaUrbano(modelo, ruido=0.02),
    "turno": 0,
}

slider_fila = widgets.IntSlider(min=0, max=estado_juego["grid"].shape[0]-1, description="Fila")
slider_columna = widgets.IntSlider(min=0, max=estado_juego["grid"].shape[1]-1, description="Columna")
selector_accion = widgets.Dropdown(options=list(ACCIONES_DISPONIBLES.keys()), description="Acción")
boton_aplicar = widgets.Button(description="Aplicar acción", button_style="info")
boton_turno = widgets.Button(description="⏭️ Avanzar turno", button_style="success")
salida = widgets.Output()

def refrescar_pantalla():
    with salida:
        clear_output(wait=True)
        dibujar_mapa(estado_juego["grid"], f"Turno {estado_juego['turno']} — {modelo['equipo']}")
        print("Indicadores actuales:")
        for nombre, valor in estado_juego["eco"].estado.items():
            print(f"  {nombre}: {valor:.1f}")

def on_aplicar_click(b):
    fila, col = slider_fila.value, slider_columna.value
    nuevo_tipo = ACCIONES_DISPONIBLES[selector_accion.value]
    estado_juego["grid"][fila, col] = nuevo_tipo
    refrescar_pantalla()

def on_turno_click(b):
    variables_mapa = variables_desde_grid(estado_juego["grid"])
    acciones = {nombre: (valor - estado_juego["eco"].estado[nombre]) * 0.3 for nombre, valor in variables_mapa.items()}
    estado_juego["eco"].paso(acciones=acciones)
    estado_juego["turno"] += 1
    refrescar_pantalla()

boton_aplicar.on_click(on_aplicar_click)
boton_turno.on_click(on_turno_click)

controles = widgets.VBox([
    widgets.HBox([slider_fila, slider_columna]),
    widgets.HBox([selector_accion, boton_aplicar, boton_turno]),
    salida,
])

display(controles)
refrescar_pantalla()


## 5. Prueben su prototipo (y su predicción)

Con su equipo:
1. Muevan los sliders hasta una celda de vegetación y cambien su acción a **"🏭 Poner fábrica"**. Denle clic a "Aplicar acción".
2. Denle clic a **"⏭️ Avanzar turno"** un par de veces.
3. Observen cómo cambiaron los indicadores.

> 📝 Nota para quien use este notebook en modo "solo ejecutar todo" (nbconvert): los controles interactivos requieren un kernel corriendo con Jupyter/Colab abierto en el navegador para hacer clic — al ejecutar el notebook completo de forma automática, verán el mapa inicial dibujado una sola vez.

### 🔍 Observen y reflexionen

**✏️ Comparen lo que pasó con su predicción de la Sección 3. Escriban su respuesta:**

- ¿Se cumplió su hipótesis sobre `vegetacion`, `contaminacion` y `agua`? *(reemplazar)*
- ¿El mapa y los indicadores se comportan como esperaban según su diagrama causal del Notebook 1? *(reemplazar)*
- ¿Notaron algún retraso entre "aplicar una acción" y "ver el efecto en los indicadores"? ¿Por qué creen que pasa eso (revisen cómo funciona `on_turno_click`)? *(reemplazar)*


## 6. (Opcional) Prototipo equivalente en Pygame para computadora local

Si alguien de su equipo tiene Python instalado localmente (no en Colab) y quiere explorar Pygame, esta celda genera un archivo `pygame_prototipo.py` con la misma idea (mapa + clics para construir), usando ventanas reales y el mouse.

Para correrlo: instalen Pygame (`pip install pygame`) y ejecuten `python pygame_prototipo.py` desde la terminal — **no desde Jupyter**.


In [ ]:
codigo_pygame = '''
# Prototipo equivalente en Pygame (para ejecutar LOCALMENTE, no en Colab/Jupyter)
# Instalar antes: pip install pygame
# Ejecutar con:   python pygame_prototipo.py

import pygame
import random

ANCHO_CELDA = 40
N = 10
COLORES = {
    0: (59, 130, 196),   # agua
    1: (76, 175, 107),   # vegetacion
    2: (158, 158, 158),  # urbano
    3: (224, 138, 44),   # industrial
}
ACCION_ACTUAL = 1  # 1=reforestar, 2=vivienda, 3=fabrica, 0=agua

def generar_grid(n=N):
    return [[random.choice([0, 1, 1, 1, 2, 2]) for _ in range(n)] for _ in range(n)]

def dibujar(pantalla, grid):
    for fila in range(len(grid)):
        for col in range(len(grid[0])):
            color = COLORES[grid[fila][col]]
            rect = pygame.Rect(col * ANCHO_CELDA, fila * ANCHO_CELDA, ANCHO_CELDA, ANCHO_CELDA)
            pygame.draw.rect(pantalla, color, rect)
            pygame.draw.rect(pantalla, (255, 255, 255), rect, 1)

def main():
    global ACCION_ACTUAL
    pygame.init()
    pantalla = pygame.display.set_mode((N * ANCHO_CELDA, N * ANCHO_CELDA))
    pygame.display.set_caption("Prototipo - Satelites y Sims")
    grid = generar_grid()
    reloj = pygame.time.Clock()
    corriendo = True

    print("Controles: teclas 1=reforestar 2=vivienda 3=fabrica 4=agua | clic para aplicar | ESC para salir")

    while corriendo:
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                corriendo = False
            elif evento.type == pygame.KEYDOWN:
                if evento.key == pygame.K_1:
                    ACCION_ACTUAL = 1
                elif evento.key == pygame.K_2:
                    ACCION_ACTUAL = 2
                elif evento.key == pygame.K_3:
                    ACCION_ACTUAL = 3
                elif evento.key == pygame.K_4:
                    ACCION_ACTUAL = 0
                elif evento.key == pygame.K_ESCAPE:
                    corriendo = False
            elif evento.type == pygame.MOUSEBUTTONDOWN:
                x, y = pygame.mouse.get_pos()
                col, fila = x // ANCHO_CELDA, y // ANCHO_CELDA
                if 0 <= fila < N and 0 <= col < N:
                    grid[fila][col] = ACCION_ACTUAL

        pantalla.fill((0, 0, 0))
        dibujar(pantalla, grid)
        pygame.display.flip()
        reloj.tick(30)

    pygame.quit()


if __name__ == "__main__":
    main()
'''

with open("pygame_prototipo.py", "w", encoding="utf-8") as f:
    f.write(codigo_pygame)

print("✅ Archivo 'pygame_prototipo.py' generado (ejecútenlo localmente con: python pygame_prototipo.py)")


## 🧭 Bitácora de aprendizaje

**✏️ Completen cada punto como equipo:**

1. ¿Qué principio de diseño de interfaz les parece más importante para que un jugador entienda su juego sin leer un manual: los colores, los íconos, los números, o los botones? Justifiquen con lo que vivieron hoy. *(reemplazar)*
2. Este prototipo conecta "mapa" → "variables" → "indicadores" en cada turno. En sus palabras, ¿por qué es importante que esa conexión sea automática (código) y no algo que el jugador calcule a mano? *(reemplazar)*
3. Si un compañero de otro equipo probara su prototipo sin ninguna explicación, ¿qué parte del mapa o los controles lo confundiría más? *(reemplazar)*

## ✅ Producto esperado de esta sesión

- [ ] Mapa/tablero del sistema representado como grid (10×10 o el tamaño que elijan)
- [ ] Controles básicos funcionando: elegir celda, elegir acción, aplicar, avanzar turno
- [ ] Indicadores del ecosistema visibles y actualizándose con cada turno
- [ ] Predicción de la Sección 3 comparada contra el resultado real
- [ ] (Opcional) `pygame_prototipo.py` generado para quienes quieran probar en su computadora
- [ ] Bitácora de aprendizaje completa

### 🎨 Ejercicios para profundizar
1. Cambien el tamaño del mapa a 15×15 o 20×20 — ¿qué tienen que ajustar?
2. Agreguen un **quinto tipo de celda** (por ejemplo, "parque público") con su propio color y efecto en las variables.
3. Agreguen una **etiqueta de texto** sobre el mapa que muestre el turno actual y el nombre de su equipo.

**Mañana:** Notebook 4 — convertimos este prototipo en un juego real con escenarios, decisiones con consecuencias y un sistema de puntuación. 🏆
